# 뉴스 기사 인기 예측 — LGB · XGB · CatBoost 앙상블

Parrot 14기 ML 프로젝트 최종 제출 코드 (v15).

- 데이터: `../data/`에 `train.csv`, `test.csv`, `leaderboard.csv`, `sample_submission_leaderboard.csv` 배치 ([data/README.md](../data/README.md))
- 흐름: Optuna 튜닝(LGB, CatBoost) → Seed 42 / 777 각각 5-Fold 3모델 학습 → OOF 기준 가중치 최적화 → 두 Seed 블렌딩
- 아래 셀 출력은 Colab에서 실제로 실행한 결과입니다 (약 35분 소요).


In [ ]:
%pip install -q catboost optuna

In [5]:
# =====================================================================
# v15 - Optuna(LGB+CAT) + Seed 42/777 두 개 5-Fold 블렌딩
# v13보다 빠르고 새로운 조합
# 예상 시간: ~35분
# =====================================================================
from pathlib import Path

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from scipy.optimize import minimize
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. 데이터 로드
DATA_DIR = Path('../data')
OUT_DIR  = Path('../output'); OUT_DIR.mkdir(exist_ok=True)

train_df       = pd.read_csv(DATA_DIR / 'train.csv')
leaderboard_df = pd.read_csv(DATA_DIR / 'leaderboard.csv')
test_df        = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub     = pd.read_csv(DATA_DIR / 'sample_submission_leaderboard.csv')

print("train shape:", train_df.shape)

# 2. 전처리
drop_cols     = ['id']
constant_cols = [col for col in train_df.columns if train_df[col].nunique() == 1]
drop_cols.extend([col for col in constant_cols if col != 'is_popular'])

X      = train_df.drop(columns=drop_cols + ['is_popular'])
y      = train_df['is_popular']
X_lb   = leaderboard_df.drop(columns=[c for c in drop_cols if c in leaderboard_df.columns])
X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])

pos_weight = (y == 0).sum() / (y == 1).sum()
print(f"피처 수: {X.shape[1]}, pos_weight: {pos_weight:.2f}")

# 3. Optuna (3-Fold, 빠르게)
print("\n=== Optuna LGB 튜닝 (20 trials) ===")
def lgb_obj(trial):
    p = dict(objective='binary', metric='average_precision', boosting_type='gbdt',
             is_unbalance=True, n_estimators=2000, random_state=42, verbose=-1, n_jobs=-1,
             learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
             num_leaves        = trial.suggest_int('num_leaves', 31, 127),
             min_child_samples = trial.suggest_int('min_child_samples', 15, 60),
             subsample         = trial.suggest_float('subsample', 0.7, 0.95),
             colsample_bytree  = trial.suggest_float('colsample_bytree', 0.7, 0.95),
             reg_alpha         = trial.suggest_float('reg_alpha', 0.01, 1.0, log=True),
             reg_lambda        = trial.suggest_float('reg_lambda', 0.1, 2.0, log=True))
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr, val in skf.split(X, y):
        m = lgb.LGBMClassifier(**p)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[val], y.iloc[val])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(False)])
        scores.append(average_precision_score(y.iloc[val], m.predict_proba(X.iloc[val])[:,1]))
    return np.mean(scores)

lgb_study = optuna.create_study(direction='maximize')
lgb_study.enqueue_trial({'learning_rate':0.03,'num_leaves':63,'min_child_samples':30,
                          'subsample':0.8,'colsample_bytree':0.8,'reg_alpha':0.05,'reg_lambda':0.5})
lgb_study.optimize(lgb_obj, n_trials=20, show_progress_bar=True)
print(f"LGB Best: {lgb_study.best_value:.5f}")

print("\n=== Optuna CAT 튜닝 (15 trials) ===")
def cat_obj(trial):
    p = dict(iterations=1500, auto_class_weights='Balanced', eval_metric='PRAUC',
             early_stopping_rounds=50, random_seed=42, verbose=0,
             learning_rate       = trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
             depth               = trial.suggest_int('depth', 5, 8),
             l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
             bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 1.0),
             random_strength     = trial.suggest_float('random_strength', 0.5, 2.0))
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr, val in skf.split(X, y):
        m = CatBoostClassifier(**p)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=(X.iloc[val], y.iloc[val]))
        scores.append(average_precision_score(y.iloc[val], m.predict_proba(X.iloc[val])[:,1]))
    return np.mean(scores)

cat_study = optuna.create_study(direction='maximize')
cat_study.enqueue_trial({'learning_rate':0.03,'depth':6,'l2_leaf_reg':3.0,
                          'bagging_temperature':0.5,'random_strength':1.0})
cat_study.optimize(cat_obj, n_trials=15, show_progress_bar=True)
print(f"CAT Best: {cat_study.best_value:.5f}")

# Optuna 최적 파라미터
opt_lgb = {**lgb_study.best_params,
           'objective':'binary', 'metric':'average_precision', 'boosting_type':'gbdt',
           'is_unbalance':True, 'n_estimators':3000, 'verbose':-1, 'n_jobs':-1}
opt_cat = {**cat_study.best_params,
           'iterations':3000, 'auto_class_weights':'Balanced',
           'eval_metric':'PRAUC', 'early_stopping_rounds':100, 'verbose':0}

base_xgb = dict(objective='binary:logistic', eval_metric='aucpr',
                scale_pos_weight=pos_weight, learning_rate=0.03, n_estimators=3000,
                max_depth=6, min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
                reg_alpha=0.05, reg_lambda=0.5, verbosity=0, n_jobs=-1)

# 4. Seed 42 / 777 각각 5-Fold 학습
def run_5fold(seed, lgb_params, xgb_params, cat_params):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof_lgb = np.zeros(len(X)); oof_xgb = np.zeros(len(X)); oof_cat = np.zeros(len(X))
    lb_lgb  = np.zeros(len(X_lb));  lb_xgb  = np.zeros(len(X_lb));  lb_cat  = np.zeros(len(X_lb))
    t_lgb   = np.zeros(len(X_test)); t_xgb  = np.zeros(len(X_test)); t_cat  = np.zeros(len(X_test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        m = lgb.LGBMClassifier(**lgb_params, random_state=seed)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof_lgb[val_idx] = m.predict_proba(X_val)[:,1]
        lb_lgb += m.predict_proba(X_lb)[:,1]   / 5
        t_lgb  += m.predict_proba(X_test)[:,1] / 5

        m = xgb.XGBClassifier(**xgb_params, random_state=seed, early_stopping_rounds=100)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb[val_idx] = m.predict_proba(X_val)[:,1]
        lb_xgb += m.predict_proba(X_lb)[:,1]   / 5
        t_xgb  += m.predict_proba(X_test)[:,1] / 5

        m = CatBoostClassifier(**cat_params, random_seed=seed)
        m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        oof_cat[val_idx] = m.predict_proba(X_val)[:,1]
        lb_cat += m.predict_proba(X_lb)[:,1]   / 5
        t_cat  += m.predict_proba(X_test)[:,1] / 5

        print(f"  Fold {fold} | LGB:{average_precision_score(y_val,oof_lgb[val_idx]):.5f}"
              f" XGB:{average_precision_score(y_val,oof_xgb[val_idx]):.5f}"
              f" CAT:{average_precision_score(y_val,oof_cat[val_idx]):.5f}")

    def neg(w):
        w = np.array(w); w = w/w.sum()
        return -average_precision_score(y, w[0]*oof_lgb+w[1]*oof_xgb+w[2]*oof_cat)
    res = minimize(neg, [1/3,1/3,1/3], method='Nelder-Mead')
    w = np.array(res.x); w = w/w.sum()

    lb_pred   = w[0]*lb_lgb  + w[1]*lb_xgb  + w[2]*lb_cat
    test_pred = w[0]*t_lgb   + w[1]*t_xgb   + w[2]*t_cat
    oof_pred  = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cat
    score = average_precision_score(y, oof_pred)
    print(f"  → OOF: {score:.5f} | LGB:{w[0]:.3f} XGB:{w[1]:.3f} CAT:{w[2]:.3f}")
    return lb_pred, test_pred, oof_pred, score

print("\n=== Seed 42 - Optuna 파라미터로 5-Fold ===")
lb_42, test_42, oof_42, score_42 = run_5fold(42, opt_lgb, base_xgb, opt_cat)

print("\n=== Seed 777 - Optuna 파라미터로 5-Fold ===")
lb_777, test_777, oof_777, score_777 = run_5fold(777, opt_lgb, base_xgb, opt_cat)

# 5. 두 seed 최적 블렌딩
print("\n=== Seed 42 + 777 최적 블렌딩 ===")
print(f"Seed 42 OOF:  {score_42:.5f}")
print(f"Seed 777 OOF: {score_777:.5f}")

for w42 in [0.3, 0.4, 0.5, 0.6, 0.7]:
    blend = w42*oof_42 + (1-w42)*oof_777
    print(f"  42:{w42:.1f} + 777:{1-w42:.1f} → OOF: {average_precision_score(y, blend):.5f}")

def neg_final(w):
    return -average_precision_score(y, w[0]*oof_42 + (1-w[0])*oof_777)
res = minimize(neg_final, [0.5], method='Nelder-Mead')
best_w = float(np.clip(res.x[0], 0.1, 0.9))

lb_final   = best_w*lb_42   + (1-best_w)*lb_777
test_final = best_w*test_42 + (1-best_w)*test_777
oof_final  = best_w*oof_42  + (1-best_w)*oof_777
print(f"\n최적 비율 → Seed42: {best_w:.3f} | Seed777: {1-best_w:.3f}")
print(f"최종 OOF PR-AUC: {average_precision_score(y, oof_final):.5f}")

# 6. 제출 파일
sub_lb = sample_sub.copy()
sub_lb['is_popular'] = lb_final
sub_lb.to_csv(OUT_DIR / 'leaderboard_submission.csv', index=False)

sub_test = pd.DataFrame({'id': test_df['id'], 'is_popular': test_final})
sub_test.to_csv(OUT_DIR / 'test_submission.csv', index=False)

print("\n저장 완료!")
print(f"  - {OUT_DIR / 'leaderboard_submission.csv'}")
print(f"  - {OUT_DIR / 'test_submission.csv'}")

train shape: (23644, 60)
피처 수: 58, pos_weight: 4.95

=== Optuna LGB 튜닝 (20 trials) ===


  0%|          | 0/20 [00:00<?, ?it/s]

LGB Best: 0.32407

=== Optuna CAT 튜닝 (15 trials) ===


  0%|          | 0/15 [00:00<?, ?it/s]

CAT Best: 0.32653

=== Seed 42 - Optuna 파라미터로 5-Fold ===
  Fold 1 | LGB:0.31206 XGB:0.30577 CAT:0.30309
  Fold 2 | LGB:0.34883 XGB:0.34781 CAT:0.34616
  Fold 3 | LGB:0.32573 XGB:0.32800 CAT:0.32827
  Fold 4 | LGB:0.31936 XGB:0.32180 CAT:0.32303
  Fold 5 | LGB:0.34382 XGB:0.34247 CAT:0.34032
  → OOF: 0.33134 | LGB:0.370 XGB:0.293 CAT:0.337

=== Seed 777 - Optuna 파라미터로 5-Fold ===
  Fold 1 | LGB:0.32086 XGB:0.31980 CAT:0.31997
  Fold 2 | LGB:0.33098 XGB:0.32509 CAT:0.33292
  Fold 3 | LGB:0.34441 XGB:0.34369 CAT:0.35049
  Fold 4 | LGB:0.33034 XGB:0.31898 CAT:0.32444
  Fold 5 | LGB:0.31500 XGB:0.31433 CAT:0.32185
  → OOF: 0.33012 | LGB:0.367 XGB:-0.022 CAT:0.654

=== Seed 42 + 777 최적 블렌딩 ===
Seed 42 OOF:  0.33134
Seed 777 OOF: 0.33012
  42:0.3 + 777:0.7 → OOF: 0.33273
  42:0.4 + 777:0.6 → OOF: 0.33318
  42:0.5 + 777:0.5 → OOF: 0.33320
  42:0.6 + 777:0.4 → OOF: 0.33315
  42:0.7 + 777:0.3 → OOF: 0.33291

최적 비율 → Seed42: 0.498 | Seed777: 0.502
최종 OOF PR-AUC: 0.33321

저장 완료!
  - /content/leader